# Restaurant Monthly Revenue Prediction (Fixed)**Fix applied:** removed `revenue_factore`, a feature that was built directly from the target (`Monthly_Revenue * Number_of_Customers`). That caused data leakage and an inflated R² score. This version reports honest performance.

In [ ]:
import pandas as pdimport numpy as npfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScaler, LabelEncoderfrom sklearn.linear_model import LinearRegressionfrom sklearn.pipeline import Pipelinefrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scoreimport matplotlib.pyplot as pltimport seaborn as sns

In [ ]:
revenue_df = pd.read_csv("Restaurant_revenue (1).csv")revenue_df.head()

In [ ]:
revenue_df.info()

In [ ]:
revenue_df.describe()

## Data quality checkA few rows have negative `Monthly_Revenue`, which is unusual for a revenue figure. Worth investigating as a possible data entry issue in a real project — flagged here, not removed.

In [ ]:
print('Negative revenue rows:', (revenue_df['Monthly_Revenue'] < 0).sum())revenue_df[revenue_df['Monthly_Revenue'] < 0]

In [ ]:
plt.figure(figsize=(8,6))sns.heatmap(revenue_df.corr(numeric_only=True), annot=True, cmap='coolwarm')plt.title('Correlation heatmap')plt.show()

## Encoding & Feature Engineering`Cuisine_Type` is label-encoded. `market_per_spend` is a legitimate interaction feature (customers × marketing spend) — it does NOT use the target, so it's safe to keep.**Removed:** `revenue_factore` (`Monthly_Revenue * Number_of_Customers`) — this directly encoded the target and caused data leakage.

In [ ]:
le = LabelEncoder()revenue_df['Cuisine_Type'] = le.fit_transform(revenue_df['Cuisine_Type'])# Safe interaction feature (no leakage — doesn't use Monthly_Revenue)revenue_df['market_per_spend'] = revenue_df['Number_of_Customers'] * revenue_df['Marketing_Spend']revenue_df.head()

## Features and target

In [ ]:
X = revenue_df.drop(['Monthly_Revenue', 'Reviews'], axis=1)y = revenue_df['Monthly_Revenue']X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42)pipeline = Pipeline([    ('scaler', StandardScaler()),    ('model', LinearRegression())])pipeline.fit(X_train, y_train)y_pred = pipeline.predict(X_test)

In [ ]:
r2 = r2_score(y_test, y_pred)mae = mean_absolute_error(y_test, y_pred)mse = mean_squared_error(y_test, y_pred)rmse = np.sqrt(mse)print('R² Score:', r2)print('MAE:', mae)print('MSE:', mse)print('RMSE:', rmse)

## Feature influenceSince features are standardized, coefficient size roughly reflects each feature's influence on the prediction.

In [ ]:
coefs = pd.Series(pipeline.named_steps['model'].coef_, index=X.columns).sort_values(key=abs, ascending=False)coefs

## Predict on new dataUsing a DataFrame with matching column names (avoids the sklearn feature-name warning from the original notebook).

In [ ]:
new_data = pd.DataFrame([{    'Number_of_Customers': 61,    'Menu_Price': 43.117635,    'Marketing_Spend': 12.663793,    'Cuisine_Type': le.transform(['Japanese'])[0],    'Average_Customer_Spending': 36.236133,    'Promotions': 0,    'market_per_spend': 61 * 12.663793}])new_data = new_data[X.columns]prediction = pipeline.predict(new_data)print('Predicted Monthly Revenue:', prediction[0])